In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

import matplotlib.pyplot as plt
import importlib
import numpy as np

from typing import TYPE_CHECKING, Callable, Union, Optional

from VariablesClass import VariablesClass
from StructureClass import  StructureClass
from StateClass import StateClass
from EquilibriumClass import EquilibriumClass
from SupervisorClass import SupervisorClass
from config import CFG

import plot_funcs, colors, helpers_builders, learning_funcs, file_funcs, numerical_experiments

## Average training time

In [ ]:
folder = "Training\\Apr23randomPosAfterFroceExplode"
average_t = file_funcs.average_successful_train_time(folder = folder, thresh=1e-6) 
print("average_t = ", average_t)

## Non abelianity check

In [ ]:
import config
importlib.reload(config)
from config import CFG

# Non-abelian buckle check: inputs
importlib.reload(numerical_experiments)

Strctr = StructureClass(CFG, update_scheme=CFG.Train.update_scheme)
Variabs = VariablesClass(Strctr, CFG)
Sprvsr = SupervisorClass(Strctr, CFG, supress_prints=False)
Sprvsr.create_dataset(Strctr, CFG, CFG.Train.dataset_sampling, tip_pos=None, tip_angle=None)

init_buckle = helpers_builders._initiate_buckle(
    CFG.Strctr.H,
    CFG.Strctr.S,
    buckle_pattern=CFG.Train.init_buckle_pattern,
)

flat_tip_pos = np.array([Strctr.edges * Strctr.L, 0.0])
flat_tip_angle = 0.0

# Edit these values for the check.
initial_tip_pos = np.array([0.82 * Strctr.edges * Strctr.L, -0.03])
initial_tip_angle = -0.05
final_tip_pos = np.array([0.5 * Strctr.edges * Strctr.L, -0.3 * Strctr.edges * Strctr.L])
final_tip_angle = -np.pi/2
Eq_iterations = 4

print("flat pose:", flat_tip_pos, flat_tip_angle)
print("initial pose:", initial_tip_pos, initial_tip_angle)
print("final pose:", final_tip_pos, final_tip_angle)
print("initial buckle:", np.asarray(init_buckle, dtype=int).reshape(-1))

In [ ]:
# Non-abelian buckle check: warm start, then compare operation order
compress_to_tip_position = getattr(
    numerical_experiments,
    "compress_to_tip_position",
    numerical_experiments.compress_to_tip_pos,
)

def buckle_tuple(buckle_arr):
    return tuple(np.asarray(buckle_arr, dtype=int).reshape(-1))

def run_leg(label, buckle, tip_pos_i, tip_angle_i, tip_pos_f, tip_angle_f, init_pos):
    print(f"\n{label}")
    State, pos_in_t, force_in_t = compress_to_tip_position(
        Strctr,
        Variabs,
        Sprvsr,
        CFG,
        np.asarray(buckle, dtype=int).copy(),
        np.asarray(tip_pos_i, dtype=float),
        float(tip_angle_i),
        np.asarray(tip_pos_f, dtype=float),
        float(tip_angle_f),
        int(Eq_iterations),
        init_pos=None if init_pos is None else np.asarray(init_pos, dtype=float).copy(),
    )
    print("buckle:", buckle_tuple(State.buckle_arr))
    return State, pos_in_t, force_in_t

State_initial, pos_warm, force_warm = run_leg(
    "warm start: flat -> initial pose",
    init_buckle,
    flat_tip_pos,
    flat_tip_angle,
    initial_tip_pos,
    initial_tip_angle,
    init_pos=None,
)
initial_pos_arr = State_initial.pos_arr.copy()
initial_buckle_arr = State_initial.buckle_arr.copy()

State_pos, pos_pos, force_pos = run_leg(
    "path A, leg 1: move position first",
    initial_buckle_arr,
    initial_tip_pos,
    initial_tip_angle,
    final_tip_pos,
    initial_tip_angle,
    init_pos=initial_pos_arr,
)
State_pos_angle, pos_pos_angle, force_pos_angle = run_leg(
    "path A, leg 2: then move angle",
    State_pos.buckle_arr,
    final_tip_pos,
    initial_tip_angle,
    final_tip_pos,
    final_tip_angle,
    init_pos=State_pos.pos_arr,
)

State_angle, pos_angle, force_angle = run_leg(
    "path B, leg 1: move angle first",
    initial_buckle_arr,
    initial_tip_pos,
    initial_tip_angle,
    initial_tip_pos,
    final_tip_angle,
    init_pos=initial_pos_arr,
)
State_angle_pos, pos_angle_pos, force_angle_pos = run_leg(
    "path B, leg 2: then move position",
    State_angle.buckle_arr,
    initial_tip_pos,
    final_tip_angle,
    final_tip_pos,
    final_tip_angle,
    init_pos=State_angle.pos_arr,
)

buckle_pos_then_angle = buckle_tuple(State_pos_angle.buckle_arr)
buckle_angle_then_pos = buckle_tuple(State_angle_pos.buckle_arr)
non_abelian = buckle_pos_then_angle != buckle_angle_then_pos

print("\nFinal buckle after position -> angle:", buckle_pos_then_angle)
print("Final buckle after angle -> position:", buckle_angle_then_pos)
print("Non-abelian in buckle space:", non_abelian)

## Fast coverage of transitions

In [ ]:
# Hamming-transition coverage from exported training files
from collections import Counter
from pathlib import Path
import re
import pandas as pd

importlib.reload(file_funcs)
importlib.reload(plot_funcs)
importlib.reload(helpers_builders)

N_BITS = 4
folder = Path(r"Training\Apr23randomPosAfterFroceExplode")
# folder = Path(r"Training\May17Like_noFlipChain")
plots_folder = Path("efficient_transitions")
plots_folder.mkdir(exist_ok=True)
plot_all_transitions = True

transitions, per_file_transitions, per_file_loss, edge_zero_loss_count, missing_edges = file_funcs.buckle_transitions(
    folder=folder,
    only_init_and_final_buckles=False,
    omit_inverted=True,
    transition_mode="hamming",
    reciprocity=False,
)

required_hamming_transitions = {
    edge
    for edge in helpers_builders.all_possible_transitions(N_BITS)
    if helpers_builders.hamming_distance_int(*edge) == 1
}


def parse_task_bits(file_name: str) -> tuple[str, str]:
    """Extract initial and desired buckle strings from a final_loss filename."""
    init_match = re.search(r"init_([01]+)", file_name)
    desired_match = re.search(r"desired_?([01]+)", file_name)
    if init_match is None or desired_match is None:
        raise ValueError(f"Could not parse init/desired buckles from {file_name}")
    return init_match.group(1), desired_match.group(1)


def hamming_only(edges):
    """Keep only directed transitions between adjacent Hamming rows."""
    return [edge for edge in edges if helpers_builders.hamming_distance_int(*edge) == 1]


def save_transition_diagram(counter: Counter, missing_edges, path: Path) -> None:
    """Save a transition diagram without displaying every intermediate figure inline."""
    original_show = plt.show
    plt.show = lambda *args, **kwargs: None
    try:
        plot_funcs.plot_transition_diagram(
            counter,
            transitions_between_runs=False,
            only_reached_nodes=False,
            edge_zero_loss_count=Counter(),
            missing_edges=missing_edges,
            layout="hamming",
        )
        plt.gcf().savefig(path, dpi=200, bbox_inches="tight")
    finally:
        plt.close("all")
        plt.show = original_show


rng = np.random.default_rng(0)
available_tasks = {}
for file_name, edges in sorted(per_file_transitions.items()):
    init_bits, desired_bits = parse_task_bits(file_name)
    task_key = (init_bits, desired_bits)
    if init_bits == desired_bits or task_key in available_tasks:
        continue
    available_tasks[task_key] = (file_name, init_bits, desired_bits, hamming_only(edges))

first_task = ("0000", "1111")
task_files = []
state_visit_counts = Counter()

def states_visited_in_run(init_bits, run_edges):
    visited = {init_bits}
    for src, dst in run_edges:
        visited.add(helpers_builders.index_to_buckle(src, N_BITS))
        visited.add(helpers_builders.index_to_buckle(dst, N_BITS))
    return visited


def add_next_task(task_key):
    task = available_tasks.pop(task_key)
    file_name, init_bits, desired_bits, run_edges = task
    task_files.append(task)
    state_visit_counts.update(states_visited_in_run(init_bits, run_edges))

if first_task not in available_tasks:
    raise FileNotFoundError("Could not find a 0000 -> 1111 task to use as training task x=0.")
add_next_task(first_task)

while available_tasks:
    scores = {
        task_key: state_visit_counts[task_key[0]] + state_visit_counts[task_key[1]]
        for task_key in available_tasks
    }
    best_score = min(scores.values())
    least_visited_tasks = [task_key for task_key, score in scores.items() if score == best_score]
    next_task = least_visited_tasks[int(rng.integers(len(least_visited_tasks)))]
    add_next_task(next_task)

cumulative_transitions = Counter()
coverage_rows = []

for order, (file_name, init_bits, desired_bits, run_edges) in enumerate(task_files):
    run_counter = Counter(run_edges)
    cumulative_transitions.update(run_counter)
    observed = set(cumulative_transitions)
    missing_hamming_transitions = sorted(required_hamming_transitions - observed)

    coverage_rows.append({
        "training_task": order,
        "file": file_name,
        "init_buckle": init_bits,
        "desired_buckle": desired_bits,
        "run_hamming_transition_events": int(sum(run_counter.values())),
        "run_unique_hamming_transitions": int(len(run_counter)),
        "cumulative_hamming_transition_events": int(sum(cumulative_transitions.values())),
        "cumulative_unique_hamming_transitions": int(len(observed)),
        "missing_hamming_transitions": int(len(missing_hamming_transitions)),
    })

    if plot_all_transitions:
        png_path = plots_folder / f"{order}_init_{init_bits}_desired_{desired_bits}.png"
        save_transition_diagram(cumulative_transitions, missing_hamming_transitions, png_path)

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv("cumulative_transition.csv", index=False)


In [ ]:
importlib.reload(plot_funcs)

plot_funcs.plot_cumulative_transition_curve(
    coverage_df,
    save_path=plots_folder / "cumulative_transition.png",
)

observed_hamming_transitions = set(cumulative_transitions)
missing_hamming_transitions = sorted(required_hamming_transitions - observed_hamming_transitions)
print(
    f"Hamming coverage: {len(observed_hamming_transitions)}/{len(required_hamming_transitions)} "
    f"directed one-bit transitions; missing {len(missing_hamming_transitions)}."
)
print("Saved cumulative CSV to cumulative_transition.csv")
print(f"Saved transition PNGs to {plots_folder}")

# coverage_df